In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set seed so data is same every time we run
np.random.seed(42)
random.seed(42)

print("Libraries loaded successfully")
print("Now generating 50,000 PhonePe users...")

Libraries loaded successfully
Now generating 50,000 PhonePe users...


In [2]:
# Define states weighted by real population
states = [
    'maharashtra', 'uttar-pradesh', 'karnataka', 'tamil-nadu',
    'rajasthan', 'gujarat', 'andhra-pradesh', 'west-bengal',
    'madhya-pradesh', 'telangana', 'bihar', 'kerala',
    'punjab', 'haryana', 'odisha', 'jharkhand',
    'assam', 'uttarakhand', 'himachal-pradesh', 'delhi'
]

state_weights = [
    0.12, 0.11, 0.09, 0.08, 0.07, 0.07, 0.06, 0.06,
    0.05, 0.05, 0.04, 0.04, 0.03, 0.03, 0.03, 0.02,
    0.02, 0.01, 0.01, 0.01
]

# Device brands weighted by real Pulse data
device_brands = ['Xiaomi', 'Samsung', 'Vivo', 'Oppo', 'Realme', 'Apple', 'Motorola', 'OnePlus', 'Others']
device_weights = [0.22, 0.20, 0.16, 0.13, 0.10, 0.05, 0.05, 0.04, 0.05]

# Age groups
age_groups = ['18-25', '26-35', '36-45', '46-60']
age_weights = [0.30, 0.40, 0.20, 0.10]

print("States defined:", len(states))
print("Device brands defined:", len(device_brands))
print("Configuration ready!")

States defined: 20
Device brands defined: 9
Configuration ready!


In [4]:
# Generate 50,000 users
n_users = 50000

# Generate signup dates between Jan 2019 to Dec 2022
start_date = datetime(2019, 1, 1)
end_date = datetime(2022, 12, 31)
date_range = (end_date - start_date).days

user_ids = [f'U{str(i).zfill(5)}' for i in range(1, n_users + 1)]
signup_dates = [start_date + timedelta(days=random.randint(0, date_range)) for _ in range(n_users)]

# KYC completed — 78% users complete KYC
kyc_completed = np.random.choice([1, 0], size=n_users, p=[0.78, 0.22])

# Bank linked — 71% of KYC users link bank
bank_linked = [1 if kyc == 1 and random.random() < 0.71 else 0 for kyc in kyc_completed]

# First transaction date — 60% of bank linked users transact within 7 days
first_txn_dates = []
for i in range(n_users):
    if bank_linked[i] == 1:
        if random.random() < 0.60:
            days_to_txn = random.randint(1, 7)
        else:
            days_to_txn = random.randint(8, 30)
        first_txn_dates.append(signup_dates[i] + timedelta(days=days_to_txn))
    else:
        first_txn_dates.append(None)

# Cashback given to 40% users
cashback_given = np.random.choice([1, 0], size=n_users, p=[0.40, 0.60])

# Churn date — 35% users churn within 90 days
churn_dates = []
for i in range(n_users):
    if first_txn_dates[i] is not None and random.random() < 0.35:
        churn_days = random.randint(30, 90)
        churn_dates.append(first_txn_dates[i] + timedelta(days=churn_days))
    else:
        churn_dates.append(None)

# Build users dataframe
df_users = pd.DataFrame({
    'user_id': user_ids,
    'signup_date': signup_dates,
    'state': np.random.choice(states, size=n_users, p=state_weights),
    'device_brand': np.random.choice(device_brands, size=n_users, p=device_weights),
    'age_group': np.random.choice(age_groups, size=n_users, p=age_weights),
    'kyc_completed': kyc_completed,
    'bank_linked': bank_linked,
    'first_txn_date': first_txn_dates,
    'cashback_given': cashback_given,
    'churn_date': churn_dates
})

print("Users generated:", len(df_users))
print("\nSample:")
print(df_users.head())
print("\nKYC rate:", round(df_users['kyc_completed'].mean() * 100, 1), "%")
print("Bank link rate:", round(df_users['bank_linked'].mean() * 100, 1), "%")
print("Cashback users:", round(df_users['cashback_given'].mean() * 100, 1), "%")

Users generated: 50000

Sample:
  user_id signup_date           state device_brand age_group  kyc_completed  \
0  U00001  2022-08-02  andhra-pradesh       Xiaomi     18-25              1   
1  U00002  2019-08-17         gujarat         Oppo     26-35              0   
2  U00003  2019-02-21      tamil-nadu       Realme     18-25              1   
3  U00004  2020-07-17         gujarat       Realme     26-35              1   
4  U00005  2020-05-16      tamil-nadu        Apple     18-25              1   

   bank_linked first_txn_date  cashback_given churn_date  
0            0            NaT               0        NaT  
1            0            NaT               0        NaT  
2            1     2019-02-26               1        NaT  
3            0            NaT               0        NaT  
4            1     2020-05-17               0        NaT  

KYC rate: 78.2 %
Bank link rate: 55.7 %
Cashback users: 39.8 %


In [5]:
# Save users to CSV
df_users.to_csv(r'C:\Users\himan\OneDrive\Desktop\phonepe-pulse-analytics\data\processed\users.csv', index=False)
print("Saved users.csv")

# Generate 200,000 transactions
merchant_categories = ['Food & Dining', 'Shopping', 'Travel', 
                        'Recharge', 'Entertainment', 'Healthcare',
                        'Education', 'Groceries', 'Fuel', 'Others']

payment_types = ['P2P', 'Merchant', 'Recharge', 'Financial Services']
payment_weights = [0.40, 0.35, 0.15, 0.10]

all_transactions = []
txn_id = 1

for _, user in df_users.iterrows():
    if user['first_txn_date'] is not None and pd.notna(user['first_txn_date']):
        
        # Number of transactions per user
        if user['cashback_given'] == 1:
            n_txns = random.randint(5, 50)
        else:
            n_txns = random.randint(1, 20)
        
        for _ in range(n_txns):
            txn_date = user['first_txn_date'] + timedelta(days=random.randint(0, 365))
            
            # Stop transactions after churn
            if user['churn_date'] is not None and pd.notna(user['churn_date']):
                if txn_date > user['churn_date']:
                    break
            
            amount = round(random.uniform(10, 10000), 2)
            cashback = round(amount * 0.02, 2) if user['cashback_given'] == 1 else 0
            success = np.random.choice([1, 0], p=[0.94, 0.06])
            
            all_transactions.append({
                'transaction_id': f'T{str(txn_id).zfill(7)}',
                'user_id': user['user_id'],
                'transaction_date': txn_date,
                'transaction_amount': amount,
                'cashback_amount': cashback,
                'merchant_category': random.choice(merchant_categories),
                'payment_type': np.random.choice(payment_types, p=payment_weights),
                'transaction_status': 'Success' if success == 1 else 'Failed'
            })
            txn_id += 1

df_transactions = pd.DataFrame(all_transactions)
df_transactions.to_csv(r'C:\Users\himan\OneDrive\Desktop\phonepe-pulse-analytics\data\processed\transactions.csv', index=False)
print("Saved transactions.csv")
print("Total transactions:", len(df_transactions))
print("\nSuccess rate:", round(df_transactions['transaction_status'].value_counts(normalize=True)['Success'] * 100, 1), "%")
print("\nPayment type breakdown:")
print(df_transactions['payment_type'].value_counts())

Saved users.csv
Saved transactions.csv
Total transactions: 314134

Success rate: 94.0 %

Payment type breakdown:
payment_type
P2P                   125456
Merchant              110060
Recharge               47122
Financial Services     31496
Name: count, dtype: int64
